# Project 4 — Notebook 1: Database Builder

Game of Thrones Lore RAG System

This notebook fetches Game of Thrones / ASOIAF articles from Wikipedia, splits them into sentences, creates embeddings for each sentence, and stores everything in an Elasticsearch index.

## Imports

In [13]:
import os
from elasticsearch import Elasticsearch, helpers
import urllib3
from pprint import pprint
import time
import nltk
from nltk.tokenize import sent_tokenize
import wikipediaapi

# Disable the specific InsecureRequestWarning
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /Users/owen_/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [14]:
from sentence_transformers import SentenceTransformer

In [15]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Connect to Elasticsearch

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

ES_USER = os.getenv('ES_USER', 'elastic')
ES_PASSWORD = os.environ['ES_PASSWORD']  # set in .env
ES_HOST = os.getenv('ES_HOST', 'https://localhost:9200/')

client = Elasticsearch(
    ES_HOST,
    basic_auth=(ES_USER, ES_PASSWORD),
    verify_certs=False,
)

In [17]:
client.info()

ObjectApiResponse({'name': 'f08d4b694cd1', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'sHThI9v4QFK889Tu6Z4A8w', 'version': {'number': '9.3.1', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '0dd66e52ba3aa076cf498264e46339dbb71f0269', 'build_date': '2026-02-23T23:37:38.684779921Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

## Fetch Wikipedia Articles

[1] Define the list of Game of Thrones Wikipedia articles to fetch. We cover characters, houses, locations, seasons, and major events.

In [20]:
ARTICLES = [
    # Main show
    ('Game of Thrones', 'show'),
    ('Game of Thrones season 1', 'season'),
    ('Game of Thrones season 2', 'season'),
    ('Game of Thrones season 3', 'season'),
    ('Game of Thrones season 4', 'season'),
    ('Game of Thrones season 5', 'season'),
    ('Game of Thrones season 6', 'season'),
    ('Game of Thrones season 7', 'season'),
    ('Game of Thrones season 8', 'season'),
    # Characters
    ('Jon Snow (character)', 'character'),
    ('Daenerys Targaryen', 'character'),
    ('Tyrion Lannister', 'character'),
    ('Cersei Lannister', 'character'),
    ('Jaime Lannister', 'character'),
    ('Arya Stark', 'character'),
    ('Sansa Stark', 'character'),
    ('Ned Stark', 'character'),
    ('Robb Stark', 'character'),
    ('Bran Stark', 'character'),
    ('Catelyn Stark', 'character'),
    ('Joffrey Baratheon', 'character'),
    ('Robert Baratheon', 'character'),
    ('Stannis Baratheon', 'character'),
    ('Renly Baratheon', 'character'),
    ('Petyr Baelish', 'character'),
    ('Varys (character)', 'character'),
    ('Samwell Tarly', 'character'),
    ('Brienne of Tarth', 'character'),
    ('Sandor Clegane', 'character'),
    ('Gregor Clegane', 'character'),
    ('Theon Greyjoy', 'character'),
    ('Yara Greyjoy', 'character'),
    ('Euron Greyjoy', 'character'),
    ('Melisandre (character)', 'character'),
    ('Davos Seaworth', 'character'),
    ('Margaery Tyrell', 'character'),
    ('Olenna Tyrell', 'character'),
    ('Tywin Lannister', 'character'),
    ('Oberyn Martell', 'character'),
    ('Missandei (character)', 'character'),
    ('Grey Worm (character)', 'character'),
    ('Jorah Mormont', 'character'),
    ('Drogo (character)', 'character'),
    ('Viserys Targaryen (character)', 'character'),
    ('Hodor (character)', 'character'),
    ('Night King (character)', 'character'),
    # Houses
    ('House Stark', 'house'),
    ('House Lannister', 'house'),
    ('House Targaryen', 'house'),
    ('House Baratheon', 'house'),
    ('House Greyjoy', 'house'),
    ('House Tyrell', 'house'),
    ('House Martell', 'house'),
    ('House Tully', 'house'),
    ('House Arryn', 'house'),
    # Locations and lore
    ('Westeros', 'lore'),
    ("King's Landing", 'lore'),
    ('Winterfell', 'lore'),
    ('The Wall (Game of Thrones)', 'lore'),
    ('Dragonstone (Game of Thrones)', 'lore'),
    ('Iron Throne (Game of Thrones)', 'lore'),
    ('Dothraki', 'lore'),
    ('White Walkers', 'lore'),
    ('Direwolf (Game of Thrones)', 'lore'),
    ('Dragons in Game of Thrones', 'lore'),
    ('Red Wedding', 'lore'),
    ('Battle of the Bastards', 'lore'),
    ('A Song of Ice and Fire', 'lore'),
]

print(f'Total articles to fetch: {len(ARTICLES)}')

Total articles to fetch: 68


[2] Fetch each article from Wikipedia and split into sentences. Sentences shorter than 20 characters are filtered out as they are typically section headers or noise.

In [19]:
wiki = wikipediaapi.Wikipedia(os.getenv('WIKI_USER_AGENT', 'got-rag-lore/0.1 (https://github.com/yourname/got-rag-lore)'), 'en')

sentences = []
sentence_id = 0
skipped = []

for title, category in ARTICLES:
    page = wiki.page(title)
    if not page.exists():
        print(f'  SKIP (not found): {title}')
        skipped.append(title)
        continue

    raw_sentences = sent_tokenize(page.text)
    raw_sentences = [s.strip() for s in raw_sentences if len(s.strip()) >= 20]

    for sent in raw_sentences:
        sentences.append({
            'sentence_id': sentence_id,
            'sentence': sent,
            'doc_title': title,
            'category': category
        })
        sentence_id += 1

    print(f'  {title}: {len(raw_sentences)} sentences')
    time.sleep(0.5)

print(f'\nTotal sentences collected: {len(sentences)}')
if skipped:
    print(f'Skipped articles: {skipped}')

  Game of Thrones: 493 sentences
  Game of Thrones season 1: 96 sentences
  Game of Thrones season 2: 125 sentences
  Game of Thrones season 3: 86 sentences
  Game of Thrones season 4: 108 sentences
  Game of Thrones season 5: 128 sentences
  Game of Thrones season 6: 120 sentences
  Game of Thrones season 7: 105 sentences
  Game of Thrones season 8: 140 sentences
  Jon Snow (character): 287 sentences
  Daenerys Targaryen: 313 sentences
  Tyrion Lannister: 320 sentences
  Cersei Lannister: 183 sentences
  Jaime Lannister: 245 sentences
  Arya Stark: 192 sentences
  Sansa Stark: 305 sentences
  Ned Stark: 122 sentences
  Robb Stark: 121 sentences
  Bran Stark: 125 sentences
  Catelyn Stark: 146 sentences
  Joffrey Baratheon: 90 sentences
  Robert Baratheon: 57 sentences
  Stannis Baratheon: 140 sentences
  Renly Baratheon: 43 sentences
  Petyr Baelish: 115 sentences
  SKIP (not found): Varys (character)
  Samwell Tarly: 135 sentences
  Brienne of Tarth: 147 sentences
  Sandor Clegane: 1

## Create Index

[3] Create the mapping for the index. The `sentence` field uses a custom analyzer with porter stemming. The `embedding` field is a dense vector for KNN search.

In [21]:
def construct_bulk_actions(batch):
    actions = []
    for doc in batch:
        emb = model.encode(doc['sentence'])
        doc['embedding'] = emb.tolist()
        action = {
            '_index': 'got_lore',
            '_id': doc['sentence_id'],
            '_source': doc
        }
        actions.append(action)
    return actions

[4] Create (or recreate) the index.

In [22]:
# Change to False if you don't want to overwrite an existing index
recreate = True

index_exists = client.indices.exists(index='got_lore')

if recreate:
    if index_exists:
        client.indices.delete(index='got_lore')
    client.indices.create(index='got_lore', body=analyzer_config)
    client.indices.put_mapping(index='got_lore', body=mapping_config)
else:
    if not index_exists:
        client.indices.create(index='got_lore', body=analyzer_config)
        client.indices.put_mapping(index='got_lore', body=mapping_config)
    else:
        print('Index exists and recreate==False. No action taken.')

## Index Documents

[5] Write a function to create bulk indexing actions for a batch of sentences. Includes creating the embedding for each sentence.

In [23]:
def construct_bulk_actions(batch):
    actions = []
    texts = [doc['sentence'] for doc in batch]
    embeddings = model.encode(texts)
    for doc, emb in zip(batch, embeddings):
        doc['embedding'] = emb.tolist()
        action = {
            '_index': 'got_lore',
            '_id': doc['sentence_id'],
            '_source': doc
        }
        actions.append(action)
    return actions

[6] Send the sentences to the database in batches. Pause index refresh until complete.

In [24]:
batch_size = 256

client.indices.put_settings(
    index='got_lore',
    body={'refresh_interval': '-1'})

for i in range(0, len(sentences), batch_size):
    batch = sentences[i : i + batch_size]
    actions = construct_bulk_actions(batch)
    helpers.bulk(client, actions)
    print(f'Batch complete. Total docs indexed: {i + len(batch)}.')

client.indices.put_settings(
    index='got_lore',
    body={'refresh_interval': '1s'})

client.indices.refresh(index='got_lore')

Batch complete. Total docs indexed: 256.
Batch complete. Total docs indexed: 512.
Batch complete. Total docs indexed: 768.
Batch complete. Total docs indexed: 1024.
Batch complete. Total docs indexed: 1280.
Batch complete. Total docs indexed: 1536.
Batch complete. Total docs indexed: 1792.
Batch complete. Total docs indexed: 2048.
Batch complete. Total docs indexed: 2304.
Batch complete. Total docs indexed: 2560.
Batch complete. Total docs indexed: 2816.
Batch complete. Total docs indexed: 3072.
Batch complete. Total docs indexed: 3328.
Batch complete. Total docs indexed: 3584.
Batch complete. Total docs indexed: 3840.
Batch complete. Total docs indexed: 4096.
Batch complete. Total docs indexed: 4352.
Batch complete. Total docs indexed: 4608.
Batch complete. Total docs indexed: 4864.
Batch complete. Total docs indexed: 5120.
Batch complete. Total docs indexed: 5376.
Batch complete. Total docs indexed: 5632.
Batch complete. Total docs indexed: 5888.
Batch complete. Total docs indexed: 6

ObjectApiResponse({'_shards': {'total': 2, 'successful': 1, 'failed': 0}})

[7] Verify the document count meets the 10,000 sentence requirement.

In [25]:
client.cat.count(index='got_lore')

TextApiResponse('1777499767 21:56:07 24103\n')